# 06 — Random Forest, Bagging (Stage 8)
Initial model with library defaults, then RandomizedSearchCV (30 candidates × 5 folds) over n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, class_weight.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'figures'; REP = ROOT / 'reports'

In [2]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['random_forest']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : RandomizedSearchCV (30 candidates)
CV Macro F1        : 0.7909 ± 0.0129
Train Macro F1     : 0.8157
Train - CV gap     : +0.0248
Best parameters    : {'model__class_weight': 'balanced', 'model__max_depth': 12, 'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__min_samples_split': 4, 'model__n_estimators': 300}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [3]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('Random Forest')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
Random Forest (initial),0.772,0.7779,0.7831,0.7995,0.7883,0.7842,0.0094
Random Forest,0.787,0.7801,0.7980,0.8122,0.7770,0.7909,0.0129


Top 10 candidates of the search, ranked by mean CV Macro F1:

In [4]:
res = pd.read_csv(REP / 'tuning' / 'random_forest_search_results.csv')
cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time']
res.sort_values('rank_test_score')[cols].head(10).round(4)

,param_model__class_weight,param_model__max_depth,param_model__max_features,param_model__min_samples_leaf,param_model__min_samples_split,param_model__n_estimators,mean_test_score,std_test_score,mean_train_score,mean_fit_time
19,balanced,12.0,0.5,1,4,300,0.7909,0.0129,0.8157,1.2235
5,balanced_subsample,16.0,NaN,1,4,506,0.7906,0.0184,0.9226,7.6597
14,balanced_subsample,24.0,sqrt,1,8,208,0.7855,0.0261,0.9955,1.3011
29,NaN,NaN,0.5,2,9,514,0.7836,0.0088,0.7964,4.8177
16,balanced_subsample,12.0,sqrt,3,6,430,0.7827,0.0146,0.9107,2.6491
4,NaN,NaN,0.5,3,4,563,0.7825,0.0103,0.8082,5.1996
3,NaN,NaN,NaN,2,7,452,0.7819,0.0075,0.8532,6.9010
23,NaN,24.0,0.5,3,5,212,0.7814,0.0100,0.7999,1.9802
1,NaN,8.0,0.5,3,9,572,0.7812,0.0102,0.7949,4.9800
27,NaN,NaN,sqrt,1,10,315,0.7807,0.0145,0.8129,2.1161
